In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/nagalakshmi00998y/8-10queue-pscabpart4/__notebook__.ipynb
/kaggle/input/notebooks/nagalakshmi00998y/8-10queue-pscabpart4/__output__.json
/kaggle/input/notebooks/nagalakshmi00998y/8-10queue-pscabpart4/__results__.html
/kaggle/input/notebooks/nagalakshmi00998y/8-10queue-pscabpart4/custom.css
/kaggle/input/notebooks/nagalakshmi00998y/8-10queue-pscabpart4/__results___files/__results___4_1.png
/kaggle/input/notebooks/nagalakshmi00998y/8-10queue-pscabpart4/__results___files/__results___4_2.png
/kaggle/input/notebooks/nagalakshmi00998y/8-10queue-pscabpart4/__results___files/__results___4_3.png
/kaggle/input/notebooks/nagalakshmi00998y/8-10queue-pscabpart4/pscab_q1/FINAL_EVIDENCE_SUMMARY.txt
/kaggle/input/notebooks/nagalakshmi00998y/8-10queue-pscabpart4/pscab_q1/PAPER_ASSETS.md
/kaggle/input/notebooks/nagalakshmi00998y/8-10queue-pscabpart4/pscab_q1/PREREGISTRATION_v2.json
/kaggle/input/notebooks/nagalakshmi00998y/8-10queue-pscabpart4/pscab_q1/RESULTS_MARKER.json
/kaggle

In [2]:
"""
================================================================================
PART 7  -  Modern baselines + TOST equivalence + tokenizer audit   (continuation, NO retraining)
================================================================================
Run in a NEW Kaggle notebook, as ONE cell, in a FRESH session (run nothing else first).
  Settings: Accelerator = GPU T4 x1, Internet = ON.
  + Add Input: the newest output of your PS-CAB notebook (the one that contains pscab_q1/
    with preds/ and analysis/). The script finds it by itself.
It installs packages and does all heavy work in a separate Python process, so the
"transformers / numpy half-upgraded" errors cannot happen.

Order (results are saved after each step, so you get the fast tables first):
  A. tokenizer audit                  (~2 min)
  B. TOST equivalence, PS-CAB vs baseline, 3 seeds, test + 3 external sets  (~3 min, CPU)
  C. zero-shot NLLB-200-600M and IndicTrans2-200M on test + 3 external sets, BLEU/chrF++/COMET (~60-90 min)
Outputs: /kaggle/working/part7/*.csv   -> Save Version when it finishes.
================================================================================
"""
import os, sys, glob, json, subprocess, importlib.metadata as md

OUT = "/kaggle/working/part7"
TMP = "/tmp/part7"
os.makedirs(OUT, exist_ok=True); os.makedirs(TMP, exist_ok=True)

def sh(cmd, check=True):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0 and check:
        print(r.stdout[-2500:]); print(r.stderr[-2500:])
        raise RuntimeError("command failed: " + " ".join(cmd[:8]))
    return r

# ---- find pscab_q1 in the attached input
cands = []
for depth in range(1, 7):
    cands += glob.glob("/kaggle/input/" + "/".join(["*"] * depth) + "/pscab_q1/preds")
if not cands:
    raise FileNotFoundError("No .../pscab_q1/preds under /kaggle/input. Attach your PS-CAB notebook output (+ Add Input).")
PRED_DIR = max(cands, key=lambda p: len(glob.glob(p + "/*.csv")))
ANA_DIR = os.path.join(os.path.dirname(PRED_DIR), "analysis")
print("[input] preds   :", PRED_DIR, f"({len(glob.glob(PRED_DIR + '/*.csv'))} files)")
print("[input] analysis:", ANA_DIR)

gpu = sh(["nvidia-smi", "-L"], check=False)
print("[env] GPU:", gpu.stdout.strip() if gpu.returncode == 0 else "NONE  (turn on GPU T4!)")

# ---- install without touching numpy/torch/scipy (pip constraints)
frozen = []
for p in ["numpy", "scipy", "pandas", "scikit-learn", "protobuf", "torch", "torchvision", "torchaudio", "pillow", "pyarrow"]:
    try: frozen.append(f"{p}=={md.version(p)}")
    except md.PackageNotFoundError: pass
cons = f"{TMP}/constraints.txt"; open(cons, "w").write("\n".join(frozen) + "\n")
PIP = [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "-c", cons]
print("[pip] transformers 4.48.3 + COMET deps ...")
sh(PIP + ["transformers==4.48.3", "sentencepiece", "sacrebleu", "entmax", "jsonargparse", "pytorch-lightning",
          "torchmetrics", "huggingface_hub", "pyyaml", "psutil", "packaging", "sacremoses"])
sh(PIP + ["--no-deps", "unbabel-comet"])
print("[pip] IndicTransToolkit (optional; IndicTrans2 is skipped if this fails) ...")
sh(PIP + ["indic-nlp-library-IT2", "cython"], check=False)
r = sh(PIP + ["--no-deps", "IndicTransToolkit"], check=False)
print("      ok" if r.returncode == 0 else "      failed -> IndicTrans2 will be skipped")

WORKER = r'''
import os, sys, re, glob, json, time, traceback
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"; os.environ["TOKENIZERS_PARALLELISM"] = "false"
import numpy as np, pandas as pd, torch, sacrebleu
from sacrebleu.metrics import CHRF, BLEU
cfg = json.load(open(sys.argv[1])); PRED, ANA, OUT = cfg["pred"], cfg["ana"], cfg["out"]
dev = "cuda" if torch.cuda.is_available() else "cpu"
SEEDS = [42, 123, 2024]
SPLITS = ["test", "extfull_FLORES200_devtest", "extfull_IN22_Gen", "extfull_IN22_Conv"]
SPLITS = [s for s in SPLITS if os.path.exists(f"{PRED}/baseline_s42__{s}.csv")]
print("[splits]", SPLITS, flush=True)

def load(run, split):
    p = f"{PRED}/{run}__{split}.csv"
    if not os.path.exists(p): return None
    d = pd.read_csv(p, dtype=str, keep_default_na=False)
    return d["en"].tolist(), d["te"].tolist(), d["prediction"].tolist()

# ======================= A. tokenizer audit =======================
from transformers import AutoTokenizer
SUF = sorted(["లోని", "లో", "కు", "కి", "తో", "నుండి", "నుంచి", "మీద", "కోసం", "వద్ద", "ద్వారా", "గురించి", "పై"], key=len, reverse=True)
_, refs0, _ = load("baseline_s42", "test")
words = [w.strip(".,;:!?\"'()[]") for s in refs0 for w in s.split()]
words = [w for w in words if w]
TOKS = {"mBART-50": "facebook/mbart-large-50-many-to-many-mmt",
        "NLLB-200": "facebook/nllb-200-distilled-600M",
        "IndicTrans2": "ai4bharat/indictrans2-en-indic-dist-200M"}
aud = []
for name, path in TOKS.items():
    try:
        tk = AutoTokenizer.from_pretrained(path, trust_remote_code=True)
        if name == "IndicTrans2" and not hasattr(tk, "tokenize"): raise RuntimeError("no tokenize()")
    except Exception as e:
        print(f"[audit] skip {name}: {str(e)[:120]}", flush=True); continue
    hit = tot = 0; fert = []
    for w in words:
        pcs = [p.replace("▁", "") for p in tk.tokenize(w)]; pcs = [p for p in pcs if p]
        fert.append(len(pcs))
        sf = next((s for s in SUF if w.endswith(s) and len(w) > len(s)), None)
        if sf:
            tot += 1
            hit += int(any("".join(pcs[k:]) == sf for k in range(1, len(pcs))))
    aud.append(dict(tokenizer=name, target_words=len(words), words_with_case_marker=tot,
                    marker_is_separate_token_pct=round(100 * hit / max(tot, 1), 1),
                    tokens_per_word=round(float(np.mean(fert)), 2)))
    print("[audit]", aud[-1], flush=True)
pd.DataFrame(aud).to_csv(f"{OUT}/T7a_tokenizer_audit.csv", index=False)

# ======================= B. TOST equivalence =======================
chrf = CHRF(word_order=2)
def stats(h, r): return np.asarray(chrf._extract_corpus_statistics(h, [r]), dtype=np.float64)
def score_from(S): return chrf._compute_score_from_stats(list(S)).score
MARGIN, B = 0.71, 2000      # 0.71 chrF = Kocmi et al. (ACL 2024) delta equivalent to 1 BLEU (~64% human agreement)
rng = np.random.default_rng(12345)
eq = []
for comp in ["pscab", "pscab_all_tokens", "pscab_random_mask", "capm"]:
    for sp in SPLITS:
        pairs = []
        for s in SEEDS:
            a, b = load(f"baseline_s{s}", sp), load(f"{comp}_s{s}", sp)
            if a and b: pairs.append((stats(a[2], a[1]), stats(b[2], b[1])))
        if not pairs: continue
        n = pairs[0][0].shape[0]
        obs = np.mean([score_from(pb.sum(0)) - score_from(pa.sum(0)) for pa, pb in pairs])
        d = np.empty(B)
        for i in range(B):
            idx = rng.integers(0, n, n)
            d[i] = np.mean([score_from(pb[idx].sum(0)) - score_from(pa[idx].sum(0)) for pa, pb in pairs])
        lo90, hi90 = np.percentile(d, [5, 95]); lo95, hi95 = np.percentile(d, [2.5, 97.5])
        eq.append(dict(system=comp, split=sp, seeds=len(pairs), n=n, delta_chrFpp=round(obs, 3),
                       CI95=f"[{lo95:+.2f}, {hi95:+.2f}]", CI90_TOST=f"[{lo90:+.2f}, {hi90:+.2f}]", margin=MARGIN,
                       equivalent_to_baseline=bool(lo90 > -MARGIN and hi90 < MARGIN),
                       worse_than_baseline=bool(hi95 < 0), better_than_baseline=bool(lo95 > 0)))
        print("[tost]", eq[-1], flush=True)
pd.DataFrame(eq).to_csv(f"{OUT}/T7b_TOST_equivalence.csv", index=False)

# ======================= C. zero-shot modern baselines =======================
from comet import download_model, load_from_checkpoint
comet = load_from_checkpoint(download_model("Unbabel/wmt22-comet-da", saving_directory="/tmp/comet_models"))
def comet_mean(src, hyp, ref, cache=None):
    if cache and os.path.exists(cache): return 100 * float(np.load(cache).mean())
    bs = 32
    while True:
        try:
            r = comet.predict([{"src": a, "mt": h, "ref": b} for a, h, b in zip(src, hyp, ref)],
                              batch_size=bs, gpus=1 if dev == "cuda" else 0, progress_bar=False, num_workers=2)
            return 100 * float(np.mean(r.scores))
        except RuntimeError as e:
            if "out of memory" in str(e).lower() and bs > 2: bs //= 2; torch.cuda.empty_cache()
            else: raise
def metrics(src, hyp, ref, cache=None):
    return dict(BLEU=round(sacrebleu.corpus_bleu(hyp, [ref]).score, 2),
                chrFpp=round(sacrebleu.corpus_chrf(hyp, [ref], word_order=2).score, 2),
                COMET=round(comet_mean(src, hyp, ref, cache), 2))

from transformers import AutoModelForSeq2SeqLM
def batches(xs, bs):
    order = np.argsort([-len(x) for x in xs]); out = [None] * len(xs)
    for i in range(0, len(xs), bs): yield order[i:i + bs], [xs[j] for j in order[i:i + bs]]
def run_nllb():
    tk = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M", src_lang="eng_Latn")
    md_ = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M", torch_dtype=torch.float16).to(dev).eval()
    bos = tk.convert_tokens_to_ids("tel_Telu")
    def f(srcs):
        out = [None] * len(srcs)
        for idx, b in batches(srcs, 24):
            enc = tk(b, return_tensors="pt", padding=True, truncation=True, max_length=256).to(dev)
            with torch.no_grad(): g = md_.generate(**enc, forced_bos_token_id=bos, num_beams=4, max_length=256)
            for j, t in zip(idx, tk.batch_decode(g, skip_special_tokens=True)): out[j] = t
        return out
    return f
def run_it2():
    try: from IndicTransToolkit.processor import IndicProcessor
    except ImportError: from IndicTransToolkit import IndicProcessor
    ip = IndicProcessor(inference=True)
    name = "ai4bharat/indictrans2-en-indic-dist-200M"
    tk = AutoTokenizer.from_pretrained(name, trust_remote_code=True)
    md_ = AutoModelForSeq2SeqLM.from_pretrained(name, trust_remote_code=True, torch_dtype=torch.float16).to(dev).eval()
    def f(srcs):
        out = [None] * len(srcs)
        for idx, b in batches(srcs, 24):
            pb = ip.preprocess_batch(b, src_lang="eng_Latn", tgt_lang="tel_Telu")
            enc = tk(pb, return_tensors="pt", padding="longest", truncation=True, max_length=256).to(dev)
            with torch.no_grad(): g = md_.generate(**enc, num_beams=4, max_length=256, use_cache=True)
            dec = tk.batch_decode(g, skip_special_tokens=True, clean_up_tokenization_spaces=True)
            for j, t in zip(idx, ip.postprocess_batch(dec, lang="tel_Telu")): out[j] = t
        return out
    return f

rows = []
T7c = f"{OUT}/T7c_modern_baselines.csv"
for sp in SPLITS:     # your own systems (seed 42) for the same table
    for run, label in [("zeroshot_mbart50", "mBART-50 zero-shot"), ("baseline_s42", "mBART-50 fine-tuned (seed 42)"),
                       ("pscab_s42", "PS-CAB (seed 42)"), ("capm_s42", "CAPM (seed 42)")]:
        x = load(run, sp)
        if x: rows.append(dict(system=label, split=sp, n=len(x[0]), **metrics(*x[:1], x[2], x[1], cache=f"{ANA}/comet__{run}__{sp}.npy")))
        if x: print("[ours]", rows[-1], flush=True)
pd.DataFrame(rows).to_csv(T7c, index=False)
for label, maker in [("NLLB-200-distilled-600M zero-shot", run_nllb), ("IndicTrans2-en-indic-dist-200M zero-shot", run_it2)]:
    try: fn = maker()
    except Exception as e:
        print(f"[skip] {label}: {str(e)[:300]}", flush=True); continue
    for sp in SPLITS:
        src, ref, _ = load("baseline_s42", sp)
        fp = f"{OUT}/preds__{label.split()[0]}__{sp}.csv"
        if os.path.exists(fp): hyp = pd.read_csv(fp, dtype=str, keep_default_na=False)["prediction"].tolist()
        else:
            t0 = time.time(); hyp = fn(src)
            pd.DataFrame({"en": src, "te": ref, "prediction": hyp}).to_csv(fp, index=False)
            print(f"[gen] {label} {sp} {time.time()-t0:.0f}s", flush=True)
        rows.append(dict(system=label, split=sp, n=len(src), **metrics(src, hyp, ref)))
        print("[new]", rows[-1], flush=True)
        pd.DataFrame(rows).to_csv(T7c, index=False)
    del fn; torch.cuda.empty_cache()

print("\n================ RESULTS (paste these to Claude) ================")
for f in ["T7a_tokenizer_audit", "T7b_TOST_equivalence", "T7c_modern_baselines"]:
    p = f"{OUT}/{f}.csv"
    if os.path.exists(p): print(f"\n### {f}\n" + pd.read_csv(p).to_string(index=False))
print("\nDone. Click Save Version.")
'''
open(f"{TMP}/worker.py", "w").write(WORKER)
json.dump({"pred": PRED_DIR, "ana": ANA_DIR, "out": OUT}, open(f"{TMP}/cfg.json", "w"))
env = dict(os.environ, PYTHONUNBUFFERED="1", HF_HOME="/tmp/hf_cache", TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD="1")
print("\n[run] worker starting ...\n")
proc = subprocess.Popen([sys.executable, f"{TMP}/worker.py", f"{TMP}/cfg.json"], stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, env=env)
for line in proc.stdout: print(line, end="")
proc.wait()
if proc.returncode != 0: raise RuntimeError(f"worker exited with code {proc.returncode}; see output above")

[input] preds   : /kaggle/input/notebooks/nagalakshmi00998y/8-10queue-pscabpart4/pscab_q1/preds (102 files)
[input] analysis: /kaggle/input/notebooks/nagalakshmi00998y/8-10queue-pscabpart4/pscab_q1/analysis
[env] GPU: GPU 0: Tesla T4 (UUID: GPU-0fcb9562-9b81-5dc3-b301-b9ff3897fade)
GPU 1: Tesla T4 (UUID: GPU-9f4a1fb8-695d-292f-aba9-06b4bff79572)
[pip] transformers 4.48.3 + COMET deps ...
[pip] IndicTransToolkit (optional; IndicTrans2 is skipped if this fails) ...
      ok

[run] worker starting ...

[splits] ['test', 'extfull_FLORES200_devtest', 'extfull_IN22_Gen', 'extfull_IN22_Conv']
[audit] {'tokenizer': 'mBART-50', 'target_words': 47661, 'words_with_case_marker': 6051, 'marker_is_separate_token_pct': 57.8, 'tokens_per_word': 2.26}
[audit] {'tokenizer': 'NLLB-200', 'target_words': 47661, 'words_with_case_marker': 6051, 'marker_is_separate_token_pct': 42.8, 'tokens_per_word': 2.41}
[audit] skip IndicTrans2: You are trying to access a gated repo.
Make sure to have access to it at http